# Qwen 2.5 7B on Google Colab (Drug Safety RAG, Person B: LLM layer)

This notebook serves **Qwen 2.5 7B Instruct with vLLM** (OpenAI-compatible API, spec p15) and runs
the LLM evaluation on the 10 synthetic patients. Everything else (retrieval, validator, fallback)
was built and tested on the Mac; this is the only part that needs a GPU.

**Before you start**
1. `Runtime > Change runtime type > GPU` (T4 is enough; L4 or A100 is faster).
2. On the Mac: `make llm-bundle` and keep `data/outputs/llm_bundle.zip` at hand (about 120 KB:
   the contracts and LLM code, the drug-name list, and the frozen evidence for the 10 patients).
3. **Synthetic patients only.** Colab is Google's machine; never send real patient data here.

**Run order:** cells 1-4 start the model (5-15 min the first time). Then **Part A** (run the
evaluation here, recommended) and/or **Part B** (reach the model from the Mac). Cell 9 stops it.

Status: written and checked offline on 2026-11-11 (cells compile; the evaluation was run against a
stand-in server). The first real Colab run is the test of cells 1-4: if the server fails to
start, cell 4 prints the vLLM log; send that log back.

In [ ]:
# 1. Which GPU? Pick the model build that fits.
import os
import subprocess

NO_GPU = (
    "No GPU in this session. Runtime > Change runtime type > T4 GPU > Save, then run this cell "
    "again. If T4 is greyed out, the free GPUs are busy: try again later."
)
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:  # CPU-only session: Colab installs nvidia-smi only with a GPU
    raise SystemExit(NO_GPU) from None
if out.returncode != 0:
    raise SystemExit(NO_GPU + "\n" + out.stderr)
name, mem_mb, cap = [x.strip() for x in out.stdout.splitlines()[0].split(",")]
mem_gb, cap = int(mem_mb.split()[0]) / 1024, float(cap)
print(f"GPU: {name}, {mem_gb:.0f} GB, compute capability {cap}")

if mem_gb >= 22:  # L4 / A100: full-precision weights (about 16 GB, spec p15)
    MODEL = "Qwen/Qwen2.5-7B-Instruct"
    EXTRA = ["--dtype", "auto"]
else:  # T4 (15 GB): the official 4-bit AWQ build; T4 has no bfloat16, and the faster
    # awq_marlin kernels need compute capability 8.0, so plain AWQ in float16.
    MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
    EXTRA = ["--dtype", "half", "--quantization", "awq"]
print("Model:", MODEL, EXTRA)

In [ ]:
# 2. Install vLLM (a few minutes) and the light packages the evaluation needs.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "vllm", "pydantic-settings", "langchain-openai"],
    check=True,
)
import vllm  # noqa: E402  (installed by the line above)

print("vLLM", vllm.__version__)  # write this down with the results

In [ ]:
# 3. Start the server in the background, protected by a random API key.
import secrets
import time

API_KEY = secrets.token_urlsafe(24)
MAX_MODEL_LEN = 16384  # the evaluation checked that the longest retry needs ~8,400 tokens
env = dict(os.environ)
if cap < 8.0:
    # A FlashInfer sampler switched on by force fails on T4 (sm_75); off, vLLM uses its own.
    env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
cmd = [
    "vllm",
    "serve",
    MODEL,
    "--port",
    "8000",
    "--api-key",
    API_KEY,
    "--max-model-len",
    str(MAX_MODEL_LEN),
    "--gpu-memory-utilization",
    "0.90",
    *EXTRA,
]
log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
print("started:", " ".join(c if c != API_KEY else "<key>" for c in cmd))

In [ ]:
# 4. Wait until the model answers (first run downloads ~5-15 GB of weights).
import httpx


def ready():
    try:
        r = httpx.get(
            "http://localhost:8000/v1/models",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=5,
        )
        return r.status_code == 200
    except httpx.HTTPError:
        return False


t0 = time.time()
while not ready():
    if server.poll() is not None or time.time() - t0 > 1800:
        print(open("vllm.log").read()[-6000:])
        raise RuntimeError("vLLM did not start: the log is above")
    time.sleep(10)
print(f"ready after {time.time() - t0:.0f}s")

t = time.time()
r = httpx.post(
    "http://localhost:8000/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=120,
    json={
        "model": MODEL,
        "temperature": 0.1,
        "max_tokens": 20,
        "messages": [{"role": "user", "content": 'Reply with {"ok": true} only.'}],
    },
)
print(f"{time.time() - t:.1f}s:", r.json()["choices"][0]["message"]["content"])

## Part A: run the evaluation here (recommended)

Upload `llm_bundle.zip` when asked. The run sends each of the 10 synthetic patients to Qwen, checks
every answer with the validator, retries once with the errors, and falls back to the template if
needed. It runs twice: plain, then with vLLM constraining the output to the Report schema
(`--guided`), so the two can be compared.

In [ ]:
# 5. Unpack the bundle and point the code at the local server.
import zipfile

from google.colab import files

up = files.upload()  # choose data/outputs/llm_bundle.zip from the Mac
zipfile.ZipFile(next(iter(up))).extractall("bundle")
os.chdir("/content/bundle")
os.environ.update(
    {
        "PYTHONPATH": "src",
        "MODEL_URL": "http://localhost:8000/v1",
        "MODEL_NAME": MODEL,
        "MODEL_API_KEY": API_KEY,
    }
)
p = subprocess.run(
    [sys.executable, "-m", "drug_safety.llm.client", "ping"],
    capture_output=True,
    text=True,
    env=os.environ,
)
print(p.stdout, p.stderr[-3000:])  # errors too: an empty output hides the cause
assert p.returncode == 0, "ping failed: send the output above"

In [ ]:
# 6. The evaluation: plain, then schema-guided. Prints a line as each patient finishes, then a
# summary (about 15-30 min for both on an L4). For a quick test, add "--limit", "2" to each.
for args in (["--plain", "--label", "plain"], ["--guided", "--label", "guided"]):
    p = subprocess.Popen(
        [sys.executable, "-u", "-m", "drug_safety.llm.eval_llm", "run", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ,
    )
    for line in p.stdout:  # live output, errors included
        print(line, end="")
    assert p.wait() == 0, "the run failed: see the lines above"

In [ ]:
# 7. Download the results. On the Mac, unzip into the repo folder: they land in data/outputs/.
import shutil

open("data/outputs/vllm_version.txt", "w").write(f"vllm {vllm.__version__}\n{MODEL}\n{name}\n")
shutil.make_archive("/content/llm_eval_results", "zip", "/content/bundle", "data/outputs")
files.download("/content/llm_eval_results.zip")

## Part B (optional): reach the model from the Mac

Opens a temporary public HTTPS address (Cloudflare quick tunnel, no account) in front of the
server. The API key above is required on every request; the address dies with the session.
Use synthetic patients only. On the Mac, put the three printed lines in `.env`, then run
`make llm-ping` and `make eval-llm`.

In [ ]:
# 8. Start the tunnel and print the settings for the Mac's .env.
import re

subprocess.run(
    [
        "wget",
        "-q",
        "-O",
        "/content/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ],
    check=True,
)
os.chmod("/content/cloudflared", 0o755)
tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tlog,
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
assert url, open("/content/tunnel.log").read()[-2000:]
print(f"MODEL_URL={url}/v1\nMODEL_NAME={MODEL}\nMODEL_API_KEY={API_KEY}")

In [ ]:
# 9. Stop everything (or just close the runtime).
for p in [globals().get("tunnel"), server]:
    if p is not None:
        p.terminate()
print("stopped")